# Tech0 50M - Colab / Kaggle quickstart
Trains the 50M model end-to-end: data -> pretrain -> distillation (Llama-3.2-3B) -> SFT -> 8-benchmark eval.

Runtime -> Change runtime type -> GPU (T4).

In [ ]:
# 1. Get the code
import os
if not os.path.exists('/content/tech0'):
    os.system('git clone https://github.com/Ritiksuman07/tech0.git /content/tech0')
os.chdir('/content/tech0')

In [ ]:
# 2. (Optional) persist checkpoints to Drive across 12h session caps
from google.colab import drive
drive.mount('/content/drive')
# then point run.out_dir at /content/drive/MyDrive/tech0/checkpoints/... in the configs

In [ ]:
# 3. Install deps (Colab already has a CUDA torch build)
!bash scripts/colab_bootstrap.sh

In [ ]:
# 4. Prepare tokenizer + packed token bins (cap volume for a quick pass)
!python -m tech0.data.prepare --config configs/pretrain.yaml --sources fineweb_edu cosmopedia wikipedia code --max-docs 50000

In [ ]:
# 5. Pretrain (resumable)
!python -m tech0.train.train_pretrain --config configs/pretrain.yaml

In [ ]:
# 6. Distillation: Llama-3.2-3B generates synthetic rationales, student anneals
!python -m tech0.train.distill --config configs/distill.yaml

In [ ]:
# 7. SFT
!python -m tech0.train.sft --config configs/sft.yaml

In [ ]:
# 8. Evaluate all 8 benchmarks
!python -m tech0.eval.run_all --config configs/eval.yaml